In [29]:
import pandas as pd
import numpy as np
import os
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
print("Libraries successfully imported!")

Libraries successfully imported!


In [30]:
#Load engineered dataset
matches= pd.read_csv("../data/processed/epl_matches_features.csv")
matches["Date"]=pd.to_datetime(matches["Date"])
matches=matches.sort_values("Date").reset_index(drop=True)
print("Dataset Shape: ", matches.shape)
matches.head()

Dataset Shape:  (16672, 31)


,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,MatchWeek,Date,HomeTeam,FullTimeScore,AwayTeam,Source_File,HomeGoals,...,HomeRecentGoalDifference,HomeMatchesPlayedBefore,HomeDaysSinceLastMatch,AwayRecentPoints,AwayRecentWinRate,AwayRecentGoalsScored,AwayRecentGoalsConceded,AwayRecentGoalDifference,AwayMatchesPlayedBefore,AwayDaysSinceLastMatch
0,0,0,0,1,2010-08-14,Bolton Wanderers FC,0-0,Fulham FC,epl_2010_11,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN
1,31,7,7,1,2010-08-14,Chelsea FC,6-0,West Bromwich Albion FC,epl_2010_11,6,...,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN
2,30,7,7,1,2010-08-14,Chelsea FC,6-0,West Bromwich Albion FC,epl_2010_11,6,...,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN
3,29,7,7,1,2010-08-14,Chelsea FC,6-0,West Bromwich Albion FC,epl_2010_11,6,...,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN
4,28,7,7,1,2010-08-14,Chelsea FC,6-0,West Bromwich Albion FC,epl_2010_11,6,...,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN


In [31]:
#Target:
y=matches["MatchResult"]
y.value_counts()

MatchResult
HW    7504
AW    5104
D     4064
Name: count, dtype: int64

In [32]:
#Input Features
feature_columns = [
    "HomeRecentPoints",
    "AwayRecentPoints",
    "HomeRecentWinRate",
    "AwayRecentWinRate",
    "HomeRecentGoalsScored",
    "AwayRecentGoalsScored",
    "HomeRecentGoalsConceded",
    "AwayRecentGoalsConceded",
    "HomeRecentGoalDifference",
    "AwayRecentGoalDifference",
    "HomeMatchesPlayedBefore",
    "AwayMatchesPlayedBefore",
    "HomeDaysSinceLastMatch",
    "AwayDaysSinceLastMatch"
]

X = matches[feature_columns].copy()
print("Input features shape:", X.shape)
print("Target shape:", y.shape)
X.head()
missing_values=X.isnull().sum()
print(missing_values[missing_values>0])

Input features shape: (16672, 14)
Target shape: (16672,)
HomeRecentPoints            112
AwayRecentPoints            112
HomeRecentWinRate           112
AwayRecentWinRate           112
HomeRecentGoalsScored       112
AwayRecentGoalsScored       112
HomeRecentGoalsConceded     112
AwayRecentGoalsConceded     112
HomeRecentGoalDifference    112
AwayRecentGoalDifference    112
HomeDaysSinceLastMatch      112
AwayDaysSinceLastMatch      112
dtype: int64


In [33]:
#Sort chronologically
matches=matches.sort_values("Date").reset_index(drop=True)
#Feature and Target in chronological order
X=matches[feature_columns].copy()
y=matches["MatchResult"].copy()

#Train and Test
split=int(len(matches)*0.80)
X_train=X.iloc[:split].copy()
X_test=X.iloc[split:].copy()

y_train=y.iloc[:split].copy()
y_test=y.iloc[split:].copy()
print("Train data: ", X_train)
print("Test data: ", X_test)
print("Training date range:",matches["Date"].iloc[0],"to",matches["Date"].iloc[split- 1])


Train data:         HomeRecentPoints  AwayRecentPoints  HomeRecentWinRate  \
0                   NaN               NaN                NaN   
1                   NaN               NaN                NaN   
2                   NaN               NaN                NaN   
3                   NaN               NaN                NaN   
4                   NaN               NaN                NaN   
...                 ...               ...                ...   
13332               0.8               0.8                0.2   
13333               0.8               1.4                0.2   
13334               1.4               0.8                0.4   
13335               3.0               3.0                1.0   
13336               3.0               2.0                1.0   

       AwayRecentWinRate  HomeRecentGoalsScored  AwayRecentGoalsScored  \
0                    NaN                    NaN                    NaN   
1                    NaN                    NaN                    NaN

In [34]:
numeric_pipeline=Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
X_train_processed=numeric_pipeline.fit_transform(X_train)
X_test_processed=numeric_pipeline.transform(X_test)
print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (13337, 14)
Processed testing shape: (3335, 14)


In [35]:
label_encoder=LabelEncoder()
y_train_encoded=label_encoder.fit_transform(y_train)
y_test_encoded=label_encoder.transform(y_test)
print("Classes: ", label_encoder.classes_)
y_train_encoded[:10]

Classes:  ['AW' 'D' 'HW']


array([1, 2, 1, 1, 1, 0, 0, 0, 2, 2])

In [36]:
print("Training Samples: ", len(X_train))
print("Testing Samples: ", len(X_test))
print("Training class distribution: ", y_train.value_counts())
print("Testing class distribution: ", y_test.value_counts())
print("Missing values after preprocessing:")
print(np.isnan(X_train_processed).sum())
print(np.isnan(X_test_processed).sum())

Training Samples:  13337
Testing Samples:  3335
Training class distribution:  MatchResult
HW    6097
AW    3924
D     3316
Name: count, dtype: int64
Testing class distribution:  MatchResult
HW    1407
AW    1180
D      748
Name: count, dtype: int64
Missing values after preprocessing:
0
0


In [38]:
os.makedirs("../data/processed", exist_ok=True)
X_train.to_csv("../data/processed/X_train.csv")
X_test.to_csv("../data/processed/X_test.csv")
y_train.to_csv("../data/processed/y_train.csv")
y_test.to_csv("../data/processed/y_test.csv")
print("Training and testing dataset saved.")

Training and testing dataset saved.
